In [ ]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet('../data/data_merged.parquet')

## Data for binary response model

In [3]:
# Binary classification: CHGOFF vs PIF only (exclude EXEMPT)
df_binary = df[df['loanstatus'].isin(['CHGOFF', 'PIF'])].copy()
df_binary = df_binary.sort_values('approvaldate').reset_index(drop=True)

In [4]:
def add_loan_features(df):
    """Create loan-related features for loan default prediction."""
    d = df.copy()
    # encode loan status (EXEMPT->0 for survival model right-censoring)
    d['default'] = d['loanstatus'].map({'CHGOFF': 1, 'PIF': 0, 'EXEMPT': 0})
    # Log of loan amount, third-party dollars (reduces skew)
    base_cpi = 100
    d['grossapproval'] = d['grossapproval'] / d['CPI_m'] * base_cpi
    d['thirdpartydollars'] = d['thirdpartydollars'] / d['CPI_m'] * base_cpi
    d['log_grossapproval'] = np.log1p(d['grossapproval'])
    d['log_thirdpartydollars'] = np.log1p(d['thirdpartydollars'])
    # Third-party leverage ratio (SBA portion vs third-party)
    d['thirdparty_ratio'] = (d['thirdpartydollars'] / (d['grossapproval'] + 1)).clip(upper=10)  # cap outliers
    # Approval to disbursement lag (log(days)) - proxy for processing speed
    d['disbursement_lag_days'] = np.log1p((d['firstdisbursementdate'] - d['approvaldate']).dt.days)
    # Temporal features from approval date
    d['approval_year'] = d['approvaldate'].dt.year
    d['approval_month'] = d['approvaldate'].dt.month
    d['approval_quarter'] = d['approvaldate'].dt.quarter
    # Long-term loan (over 20 years)
    d['long_term_loan'] = (d['terminmonths'] >= 240).astype(int)
    # Term an integer multiple of a year
    d['terminmonths_multiple_12'] = (d['terminmonths'] % 12 == 0).astype(int)
    # Lender state != borrower state
    d['diff_lb_state'] = (d['cdc_state'] != d['borrstate']).astype(int)
    # Project state != borrower state
    d['diff_pb_state'] = (d['projectstate'] != d['borrstate']).astype(int)
    # repeat borrower: defined as any business with more than one loan
    # l2locid_count: number of loans per l2locid
    d = d.sort_values('approvaldate')
    d['repeat_borrower'] = d.duplicated('borrname').astype(int)
    d['l2locid_count'] = d.groupby('l2locid').cumcount() + 1
    d['l2locid_count'] = d['l2locid_count'].fillna(0).astype(int)
    d = d.sort_index().drop(columns=['borrname', 'l2locid'])
    
    return d

def add_macro_features(df):
    """Add macroeconomic-related features to the dataframe."""
    d = df.copy()
    # real GDP
    d['GDP_y_st'] = d['GDP_y_st'] / d['CPI_m'] * 100
    # market stress: vix_m > 25
    d['market_stress_m'] = (d['vix_m'] > 25).astype(int)
    # yield inversion: yield_spread_m < 0
    d['yield_inversion_m'] = (d['yield_spread_m'] < 0).astype(int)

    return d

In [5]:
# exclude the states with small sample size
df_binary = df_binary[~df_binary['borrstate'].isin(['PR','VI','GU'])].copy()

In [6]:
df_binary = add_loan_features(df_binary)
df_binary = add_macro_features(df_binary)
df_binary = df_binary.drop(columns=['asofdate', 'approvaldate', 'firstdisbursementdate', 'paidinfulldate',
                                    'chargeoffdate', 'grosschargeoffamount', 'loanstatus',
                                    'grossapproval', 'thirdpartydollars', 
                                    'thirdpartylender_name', 'thirdpartylender_city', 'thirdpartylender_state',
                                    'borrcity', 'projectstate', 'cdc_state',
                                    'processingmethod', 'businesstype', 'businessage'])
print(f"Features number: {len(df_binary.columns)}")
print(df_binary.columns)

Features number: 44
Index(['borrstate', 'subprogram', 'terminmonths', 'naicscode', 'jobssupported',
       'collateralind', 'iffranchise', 'ifthirdparty', 'GDP_y_st',
       'GDP_growth_y_st', 'EHE_m_st', 'EHE_growth_m_st', 'CPI_m',
       'CPI_growth_m', 'HPI_m', 'HPI_growth_m', 'unemp_m_st',
       'unemp_growth_m_st', 'crime_rate_m_st', 'crime_rate_growth_m_st',
       'PPI_m', 'PPI_growth_m', 'sp500_ret_m', 'vix_m', 'tnx_m', 't2y_m',
       'russell2000_ret_m', 'yield_spread_m', 'default', 'log_grossapproval',
       'log_thirdpartydollars', 'thirdparty_ratio', 'disbursement_lag_days',
       'approval_year', 'approval_month', 'approval_quarter', 'long_term_loan',
       'terminmonths_multiple_12', 'diff_lb_state', 'diff_pb_state',
       'repeat_borrower', 'l2locid_count', 'market_stress_m',
       'yield_inversion_m'],
      dtype='object')


In [7]:
# get dummy variables
cat_cols = df_binary.select_dtypes(include=['string', 'object', 'category']).columns
df_binary = pd.get_dummies(df_binary, columns=cat_cols, drop_first=True)

In [ ]:
# Split by approvaldate year: Train 1990-2011, Val 2012-2014, Test 2015-2025
year = df_binary['approval_year']
df_train = df_binary[(year >= 1990) & (year <= 2011)].copy()
df_val = df_binary[(year >= 2012) & (year <= 2014)].copy()
df_test = df_binary[(year >= 2015) & (year <= 2025)].copy()

# from sklearn.model_selection import train_test_split

# # 6:2:2 random partitioning
# df_train, df_temp = train_test_split(df_binary, test_size=0.4, random_state=42, stratify=df_binary['default'])
# df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp['default'])
# print(df_train.shape, df_val.shape, df_test.shape)

# Target variable
TARGET = 'default'
y_train = (df_train[TARGET] == 1).astype(int)
y_val = (df_val[TARGET] == 1).astype(int)
y_test = (df_test[TARGET] == 1).astype(int)

print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"Train CHGOFF rate: {y_train.mean():.2%} | Val: {y_val.mean():.2%} | Test: {y_test.mean():.2%}")

Train: 97,986 | Val: 12,898 | Test: 11,380
Train CHGOFF rate: 11.57% | Val: 2.25% | Test: 1.75%


In [10]:
# Fillna: median by group. Rules: m -> groupby(year,month); y -> groupby(year)
def fillna_by_group(df_in, train, cols_with_null, method='median'):
    out = df_in.copy()

    cols_y = [c for c in cols_with_null if '_y' in c]
    cols_m = [c for c in cols_with_null if '_m' in c]

    if cols_y:
        if method == 'median':
            stat = train.groupby('approval_year')[cols_y].median()
        else:
            stat = train.groupby('approval_year')[cols_y].last()

        out = out.join(stat, on='approval_year', rsuffix='_fill')

        for c in cols_y:
            out[c] = out[c].fillna(out[c + '_fill'])
            out[c] = out[c].fillna(stat[c].iloc[-1])
            out.drop(columns=c + '_fill', inplace=True)

    if cols_m:
        if method == 'median':
            stat = train.groupby(['approval_year','approval_month'])[cols_m].median()
        else:
            stat = train.groupby(['approval_year','approval_month'])[cols_m].last()

        out = out.join(stat, on=['approval_year','approval_month'], rsuffix='_fill')

        for c in cols_m:
            out[c] = out[c].fillna(out[c + '_fill'])
            out[c] = out[c].fillna(stat[c].iloc[-1])
            out.drop(columns=c + '_fill', inplace=True)

    return out

num_cols_with_null = [c for c in df_train.select_dtypes(include='number').columns if df_train[c].isnull().any()]
df_train = fillna_by_group(df_train, df_train, num_cols_with_null)
df_val = fillna_by_group(df_val, df_train, num_cols_with_null)
df_test = fillna_by_group(df_test, df_train, num_cols_with_null)
print("Fillna done. Remaining nulls:", df_train.isnull().sum().sum(), df_val.isnull().sum().sum(), df_test.isnull().sum().sum())

Fillna done. Remaining nulls: 0 0 0


In [11]:
# Z-score: use train mean and std for train, val, test
num_cols = df_train.select_dtypes(include='number').columns.tolist()
binary_cols = [c for c in num_cols if set(df_train[c].dropna().unique()).issubset({0,1})]
num_cols = [c for c in num_cols if (c != TARGET) and (c not in binary_cols)]

train_mean = df_train[num_cols].mean()
train_std = df_train[num_cols].std()

df_train[num_cols] = (df_train[num_cols] - train_mean) / train_std
df_val[num_cols] = (df_val[num_cols] - train_mean) / train_std
df_test[num_cols] = (df_test[num_cols] - train_mean) / train_std

print("Z-score done for continuous vars (train mean/std applied to train, val, test)")

Z-score done for continuous vars (train mean/std applied to train, val, test)


In [12]:
df_train_val = pd.concat([df_train, df_val])

df_train_val.to_parquet('../data/data_binary_train_val.parquet', index=False)
df_train.to_parquet('../data/data_binary_train.parquet', index=False)
df_val.to_parquet('../data/data_binary_val.parquet', index=False)
df_test.to_parquet('../data/data_binary_test.parquet', index=False)

## Data for static survival model

In [13]:
# Survival model: time-to-default (duration + event) for CHGOFF+PIF+EXEMPT
# EXEMPT = right-censored (event=0, end_date=asofdate)
df_surv = df[df['loanstatus'].isin(['CHGOFF', 'PIF', 'EXEMPT'])].copy()
df_surv = df_surv[~df_surv['borrstate'].isin(['PR','VI','GU'])].copy()
df_surv = df_surv.sort_values('approvaldate').reset_index(drop=True)

# Add loan & macro features (keep date cols and grosschargeoffamount for now)
df_surv = add_loan_features(df_surv)
df_surv = add_macro_features(df_surv)

# duration: firstdisbursementdate -> chargeoffdate (default) or paidinfulldate (PIF) or asofdate (EXEMPT, right-censored)
# event: 1 if CHGOFF, 0 if PIF or EXEMPT (censored)
df_surv['end_date'] = np.where(
    df_surv['loanstatus'] == 'CHGOFF',
    df_surv['chargeoffdate'],
    np.where(
        df_surv['loanstatus'] == 'PIF',
        df_surv['paidinfulldate'],
        df_surv['asofdate']   # EXEMPT
    )
)
df_surv['duration_days'] = (df_surv['end_date'] - df_surv['firstdisbursementdate']).dt.days

# Sanity: duration > 0
df_surv = df_surv[df_surv['duration_days'] > 0].copy()
df_surv = df_surv[df_surv['duration_days'] <= 365*20]
df_surv['event'] = df_surv['default'].astype(int)  # 1=default, 0=censored

In [14]:
# Drop cols not needed for modeling
drop_cols_surv = ['asofdate', 'approvaldate', 'firstdisbursementdate', 'paidinfulldate', 'grossapproval', 'thirdpartydollars', 
                  'chargeoffdate', 'grosschargeoffamount', 'end_date', 'loanstatus', 
                  'thirdpartylender_name', 'thirdpartylender_city', 'thirdpartylender_state',
                  'borrcity', 'projectstate', 'cdc_state',
                  'processingmethod', 'businesstype', 'businessage']

df_surv = df_surv.drop(columns=[c for c in drop_cols_surv if c in df_surv.columns])

# One-hot encode on full survival data so train/val/test splits have same columns
cat_cols_surv = df_surv.select_dtypes(include=['string', 'object', 'category']).columns
df_surv = pd.get_dummies(df_surv, columns=cat_cols_surv, drop_first=True)

print("Survival cols:", len(df_surv.columns))

Survival cols: 120


In [15]:
# Train/Val/Test split (same year cutoffs as binary)
year_surv = df_surv['approval_year']
df_surv_train = df_surv[(year_surv >= 1990) & (year_surv <= 2011)].copy()
df_surv_val = df_surv[(year_surv >= 2012) & (year_surv <= 2014)].copy()
df_surv_test = df_surv[(year_surv >= 2015) & (year_surv <= 2025)].copy()

print("Survival Train:", len(df_surv_train), "Val:", len(df_surv_val), "Test:", len(df_surv_test))

Survival Train: 105105 Val: 19189 Test: 57399


In [16]:
# Fillna for survival/LGD (reuse fillna_by_group - only cols with _y or _m)
num_cols_surv = [c for c in df_surv_train.select_dtypes(include='number').columns 
                 if df_surv_train[c].isnull().any() and ('_y' in c or '_m' in c)]
df_surv_train = fillna_by_group(df_surv_train, df_surv_train, num_cols_surv)
df_surv_val   = fillna_by_group(df_surv_val, df_surv_train, num_cols_surv)
df_surv_test  = fillna_by_group(df_surv_test, df_surv_train, num_cols_surv)

print("Fillna done. Survival nulls:", df_surv_train.isnull().sum().sum(), df_surv_val.isnull().sum().sum(), df_surv_test.isnull().sum().sum())

Fillna done. Survival nulls: 0 0 0


In [17]:
# Z-score for survival (exclude binary, duration_days, event, default)
exclude_surv = ['default', 'duration_days', 'event']

num_cols_surv_z = df_surv_train.select_dtypes(include='number').columns.tolist()
binary_surv = [c for c in num_cols_surv_z if set(df_surv_train[c].dropna().unique()).issubset({0, 1})]
num_cols_surv_z = [c for c in num_cols_surv_z if c not in exclude_surv and c not in binary_surv]

surv_mean = df_surv_train[num_cols_surv_z].mean()
surv_std = df_surv_train[num_cols_surv_z].std()
df_surv_train[num_cols_surv_z] = (df_surv_train[num_cols_surv_z] - surv_mean) / surv_std
df_surv_val[num_cols_surv_z] = (df_surv_val[num_cols_surv_z] - surv_mean) / surv_std
df_surv_test[num_cols_surv_z] = (df_surv_test[num_cols_surv_z] - surv_mean) / surv_std

print("Z-score done for survival")

Z-score done for survival


In [18]:
# Save survival data
df_surv_train_val = pd.concat([df_surv_train, df_surv_val])

df_surv_train_val.to_parquet('../data/data_survival_train_val.parquet', index=False)
df_surv_train.to_parquet('../data/data_survival_train.parquet', index=False)
df_surv_val.to_parquet('../data/data_survival_val.parquet', index=False)
df_surv_test.to_parquet('../data/data_survival_test.parquet', index=False)

print("Saved: data_survival_*.parquet (duration_days, event)")

Saved: data_survival_*.parquet (duration_days, event)


## Data for time-varying survival model

In [19]:
df_tv = pd.read_parquet('../data/data_merged_tv.parquet')

In [20]:
df_tv.columns

Index(['asofdate', 'l2locid', 'borrname', 'borrcity', 'borrstate', 'cdc_state',
       'thirdpartylender_name', 'thirdpartylender_city',
       'thirdpartylender_state', 'thirdpartydollars', 'grossapproval',
       'approvaldate', 'firstdisbursementdate', 'processingmethod',
       'subprogram', 'terminmonths', 'naicscode', 'projectstate',
       'businesstype', 'businessage', 'loanstatus', 'paidinfulldate',
       'chargeoffdate', 'grosschargeoffamount', 'jobssupported',
       'collateralind', 'iffranchise', 'ifthirdparty', 'end_date', 'event',
       'loan_id', 'month_start', 'month_stop', 'loan_age', 'calendar_month',
       'event_t', 'panel_date', 'GDP_y_st', 'GDP_growth_y_st', 'EHE_m_st',
       'EHE_growth_m_st', 'CPI_m', 'CPI_growth_m', 'HPI_m', 'HPI_growth_m',
       'unemp_m_st', 'unemp_growth_m_st', 'crime_rate_m_st',
       'crime_rate_growth_m_st', 'PPI_m', 'PPI_growth_m', 'sp500_ret_m',
       'vix_m', 'tnx_m', 't2y_m', 'russell2000_ret_m', 'yield_spread_m'],
      dtype

In [21]:
df_surv_tv = df_tv[~df_tv['borrstate'].isin(['PR','VI','GU'])].copy()
df_surv_tv = df_surv_tv.sort_values('approvaldate').reset_index(drop=True)

# Add loan & macro features
df_surv_tv = add_loan_features(df_surv_tv)
df_surv_tv = add_macro_features(df_surv_tv)

In [22]:
# Drop cols not needed for modeling (keep duration, event, LGD as targets)
drop_cols_surv = ['asofdate', 'approvaldate', 'firstdisbursementdate', 'paidinfulldate', 
                  'chargeoffdate', 'grosschargeoffamount', 'end_date', 'loanstatus', 'panel_date',
                  'thirdpartylender_name', 'thirdpartylender_city', 'thirdpartylender_state',
                  'borrcity', 'projectstate', 'cdc_state',
                  'processingmethod', 'businesstype', 'businessage']

df_surv_tv = df_surv_tv.drop(columns=[c for c in drop_cols_surv if c in df_surv_tv.columns])
df_surv_tv.columns

Index(['borrstate', 'thirdpartydollars', 'grossapproval', 'subprogram',
       'terminmonths', 'naicscode', 'jobssupported', 'collateralind',
       'iffranchise', 'ifthirdparty', 'event', 'loan_id', 'month_start',
       'month_stop', 'loan_age', 'calendar_month', 'event_t', 'GDP_y_st',
       'GDP_growth_y_st', 'EHE_m_st', 'EHE_growth_m_st', 'CPI_m',
       'CPI_growth_m', 'HPI_m', 'HPI_growth_m', 'unemp_m_st',
       'unemp_growth_m_st', 'crime_rate_m_st', 'crime_rate_growth_m_st',
       'PPI_m', 'PPI_growth_m', 'sp500_ret_m', 'vix_m', 'tnx_m', 't2y_m',
       'russell2000_ret_m', 'yield_spread_m', 'default', 'log_grossapproval',
       'log_thirdpartydollars', 'thirdparty_ratio', 'disbursement_lag_days',
       'approval_year', 'approval_month', 'approval_quarter', 'long_term_loan',
       'terminmonths_multiple_12', 'diff_lb_state', 'diff_pb_state',
       'repeat_borrower', 'l2locid_count', 'market_stress_m',
       'yield_inversion_m'],
      dtype='object')

In [23]:
# One-hot encode on full survival data so train/val/test splits have same columns
cat_cols_surv = [col for col in df_surv_tv.select_dtypes(include=['string', 'object', 'category']).columns if col != 'loan_id']
df_surv_tv = pd.get_dummies(df_surv_tv, columns=cat_cols_surv, drop_first=True)

print("Survival cols:", len(df_surv_tv.columns))

Survival cols: 127


In [24]:
# Train/Val/Test split (same year cutoffs as binary)
year_surv = df_surv_tv['approval_year']
df_surv_tv_train = df_surv_tv[(year_surv >= 1990) & (year_surv <= 2011)].copy()
df_surv_tv_val   = df_surv_tv[(year_surv >= 2012) & (year_surv <= 2014)].copy()
df_surv_tv_test  = df_surv_tv[(year_surv >= 2015) & (year_surv <= 2025)].copy()

print("Survival Train:", len(df_surv_tv_train), "Val:", len(df_surv_tv_val), "Test:", len(df_surv_tv_test))

Survival Train: 12617398 Val: 1872435 Test: 2990679


In [25]:
tv_cols = ['GDP_y_st', 'GDP_growth_y_st', 'CPI_m', 'CPI_growth_m', 
           'crime_rate_m_st', 'crime_rate_growth_m_st', 'EHE_m_st',
           'EHE_growth_m_st', 'HPI_m', 'HPI_growth_m', 'PPI_m', 'PPI_growth_m',
           'unemp_m_st', 'unemp_growth_m_st', 'sp500_ret_m', 'vix_m', 'tnx_m',
           't2y_m', 'russell2000_ret_m', 'yield_spread_m']

# Fillna
df_surv_tv_train = fillna_by_group(df_surv_tv_train, df_surv_tv_train, tv_cols)
df_surv_tv_val = fillna_by_group(df_surv_tv_val, df_surv_tv_train, tv_cols)
df_surv_tv_test = fillna_by_group(df_surv_tv_test, df_surv_tv_train, tv_cols, method='last')

print("Fillna done. Survival nulls:", df_surv_tv_train.isnull().sum().sum(), df_surv_tv_val.isnull().sum().sum(), df_surv_tv_test.isnull().sum().sum())

Fillna done. Survival nulls: 0 0 0


In [ ]:
# Z-score for survival (exclude binary, duration_days, event_t, default, grossapproval, terminmonths)
exclude_surv = ['month_start', 'month_stop', 'event', 'event_t', 'grossapproval', 'terminmonths']

num_cols_surv_z = df_surv_tv_train.select_dtypes(include='number').columns.tolist()
binary_surv = [c for c in num_cols_surv_z if set(df_surv_tv_train[c].dropna().unique()).issubset({0, 1})]
num_cols_surv_z = [c for c in num_cols_surv_z if c not in exclude_surv and c not in binary_surv]

surv_mean = df_surv_tv_train[num_cols_surv_z].mean()
surv_std = df_surv_tv_train[num_cols_surv_z].std()
df_surv_tv_train[num_cols_surv_z] = (df_surv_tv_train[num_cols_surv_z] - surv_mean) / surv_std
df_surv_tv_val[num_cols_surv_z] = (df_surv_tv_val[num_cols_surv_z] - surv_mean) / surv_std
df_surv_tv_test[num_cols_surv_z] = (df_surv_tv_test[num_cols_surv_z] - surv_mean) / surv_std

print("Z-score done for survival")

Z-score done for survival


In [27]:
# Save survival data
df_surv_tv_train_val = pd.concat([df_surv_tv_train, df_surv_tv_val])

df_surv_tv_train_val.to_parquet('../data/data_tv_train_val.parquet', index=False)
df_surv_tv_train.to_parquet('../data/data_tv_train.parquet', index=False)
df_surv_tv_val.to_parquet('../data/data_tv_val.parquet', index=False)
df_surv_tv_test.to_parquet('../data/data_tv_test.parquet', index=False)

print("Saved: data_tv_*.parquet ")

Saved: data_tv_*.parquet 
